# 07 - Performance, Memory, and Gotchas

**Prerequisite:** notebooks 01-06.
**This notebook:** how to make Pandas fast, how to make it fit in memory, and the traps that produce wrong answers silently.

For an engineer used to the JVM, the mental shift is this: in Java you optimise by writing a tighter loop. In Pandas you optimise by **deleting the loop**.

In [ ]:
import numpy as np
import pandas as pd
import time
from pathlib import Path

DATA = next(p for p in [Path("../data"), Path("data"), Path("study/data")] if p.exists())
orders = pd.read_csv(DATA / "orders.csv", parse_dates=["order_ts"])
print("pandas", pd.__version__)

def timeit(fn, n=3, label=""):
    times = []
    for _ in range(n):
        t0 = time.perf_counter(); out = fn(); times.append(time.perf_counter() - t0)
    ms = min(times) * 1000
    print(f"{label:<34} {ms:>9.2f} ms")
    return ms

## 1. The performance ladder

Same calculation, six ways, slowest to fastest.

In [ ]:
big = pd.DataFrame({
    "a": np.random.default_rng(0).normal(100, 10, 200_000),
    "b": np.random.default_rng(1).normal(50, 5, 200_000),
})

def with_iterrows():
    out = []
    for _, row in big.iterrows():
        out.append(row["a"] * 0.9 + row["b"])
    return out

def with_itertuples():
    return [r.a * 0.9 + r.b for r in big.itertuples(index=False)]

def with_apply_axis1():
    return big.apply(lambda r: r["a"] * 0.9 + r["b"], axis=1)

def with_zip():
    return [a * 0.9 + b for a, b in zip(big["a"], big["b"])]

def with_vectorized():
    return big["a"] * 0.9 + big["b"]

def with_numpy():
    return big["a"].to_numpy() * 0.9 + big["b"].to_numpy()

t1 = timeit(with_iterrows,    1, "iterrows (worst)")
t2 = timeit(with_apply_axis1, 1, "apply(axis=1)")
t3 = timeit(with_itertuples,  3, "itertuples")
t4 = timeit(with_zip,         3, "zip over columns")
t5 = timeit(with_vectorized,  5, "vectorized Series arithmetic")
t6 = timeit(with_numpy,       5, "NumPy arrays directly")
print()
print(f"iterrows is roughly {t1 / t6:,.0f}x slower than plain NumPy")

**Why `iterrows` is so bad:** it constructs a new `Series` object for every single row, which also upcasts mixed types to `object`. `itertuples` builds a lightweight namedtuple instead and is ~10x faster, but it is still a Python loop.

**Rule:** if you find yourself writing `for ... in df`, stop and ask what the whole-column expression is.

### When the logic really is row-wise

Most "I need a loop" cases are actually maskable.

In [ ]:
# Bad: a Python function per row
def classify_slow(df):
    def f(row):
        if row["amount"] > 3000: return "platinum"
        if row["amount"] > 1500: return "gold"
        if row["discount_pct"] > 10: return "promo"
        return "standard"
    return df.apply(f, axis=1)

# Good: the same logic as vectorized conditions
def classify_fast(df):
    conds = [df["amount"] > 3000, df["amount"] > 1500, df["discount_pct"] > 10]
    labels = ["platinum", "gold", "promo"]
    return pd.Series(np.select(conds, labels, default="standard"), index=df.index)

o = pd.concat([orders] * 40, ignore_index=True)      # ~20k rows
timeit(lambda: classify_slow(o), 1, "apply(axis=1)")
timeit(lambda: classify_fast(o), 3, "np.select")
print()
print("identical results:", classify_slow(o).equals(classify_fast(o)))

### `map` on a Series is acceptable; `apply(axis=1)` rarely is

In [ ]:
s = pd.concat([orders["status"]] * 400, ignore_index=True)      # 200k rows
mapping = {"PLACED": 0, "SHIPPED": 1, "DELIVERED": 2, "CANCELLED": -1}

timeit(lambda: s.map(mapping),                3, "Series.map with a dict")
timeit(lambda: s.apply(lambda v: mapping[v]), 3, "Series.apply with a lambda")
print()
print("map with a dict does the lookup loop in C; apply calls your Python function per value.")
print("On a few hundred rows the difference is invisible - it only shows up at scale.")

## 2. `eval` and `query` on large frames

`df.eval("(a + b) * (c - d)")` and `df.query("a > 0 and b < 0")` can evaluate a whole expression without materialising every intermediate array - but **only when the optional `numexpr` package is installed**. Without it, Pandas falls back to ordinary Python evaluation and these are *slower* than plain arithmetic because of the parsing overhead.

The cell below prints whether `numexpr` is present, so read the timings accordingly. Treat `query`/`eval` as a readability tool first and a performance tool second.

In [ ]:
try:
    import numexpr
    print("numexpr is installed - eval/query use the accelerated engine\n")
except ImportError:
    print("numexpr NOT installed - eval/query fall back to Python and will look slower\n")

df = pd.DataFrame(np.random.default_rng(2).normal(size=(500_000, 4)), columns=list("abcd"))

timeit(lambda: (df["a"] + df["b"]) * (df["c"] - df["d"]),   3, "plain arithmetic")
timeit(lambda: df.eval("(a + b) * (c - d)"),                3, "df.eval")
print()
timeit(lambda: df[(df["a"] > 0) & (df["b"] < 0)],           3, "boolean mask filter")
timeit(lambda: df.query("a > 0 and b < 0"),                 3, "df.query")

## 3. Memory

Three levers, in order of payoff: **read fewer columns**, **use smaller dtypes**, **use `category` for repeated text**.

In [ ]:
o = orders.copy()
print("baseline:")
print(o.memory_usage(deep=True))
print("total:", o.memory_usage(deep=True).sum(), "bytes")

In [ ]:
def shrink(df):
    out = df.copy()
    for col in out.select_dtypes(include=["int64"]).columns:
        out[col] = pd.to_numeric(out[col], downcast="integer")
    for col in out.select_dtypes(include=["float64"]).columns:
        out[col] = pd.to_numeric(out[col], downcast="float")
    for col in out.select_dtypes(include=["str", "object"]).columns:
        if out[col].nunique() / max(len(out), 1) < 0.5:        # low cardinality
            out[col] = out[col].astype("category")
    return out

small = shrink(o)
before = o.memory_usage(deep=True).sum()
after  = small.memory_usage(deep=True).sum()
print(small.dtypes)
print()
print(f"{before:,} bytes -> {after:,} bytes  ({before / after:.1f}x smaller)")

> **Careful with `downcast="float"`.** `float32` has about 7 significant digits. For money, that is not enough beyond a few million - keep currency in `float64`, or better, in integer minor units (paise, cents) as `int64`. This is the same reasoning that makes you use `BigDecimal` instead of `double` in Java financial code.

In [ ]:
# Proof, so this is not just advice
amount = 12_345_678.91
print("float64:", repr(np.float64(amount)))
print("float32:", repr(np.float32(amount)), " <- already wrong in the paise")

### Reading files bigger than RAM

In [ ]:
# usecols is the cheapest win - you never pay for columns you do not read
few = pd.read_csv(DATA / "orders.csv", usecols=["order_id", "amount"])
print("columns read:", list(few.columns), "| memory:", few.memory_usage(deep=True).sum(), "bytes")
print()

# chunksize streams the file; aggregate as you go, never hold it all
total, rows = 0.0, 0
for chunk in pd.read_csv(DATA / "orders.csv", usecols=["amount"], chunksize=100):
    total += chunk["amount"].sum()
    rows  += len(chunk)
print(f"streamed {rows} rows in chunks, total amount = {total:,.2f}")

## 4. Copy-on-Write (Pandas 3.x) - what changed

Pandas 3.0 made **Copy-on-Write** mandatory. It cannot be turned off. The rules are now simple and worth memorising:

1. Any operation that returns a new object behaves as if it returned a **copy**. You can never accidentally modify the parent through a child.
2. **Chained assignment never works.** `df["a"][0] = 1` and `df[df.x > 0]["a"] = 1` silently update a temporary and are lost.
3. The copy is lazy - Pandas only really copies when something is written, so this is usually *faster* than the old behaviour, not slower.

If you learned Pandas before 2024 you will remember `SettingWithCopyWarning`. That warning is gone; the ambiguity it warned about is gone too.

In [ ]:
import warnings

df = pd.DataFrame({"a": [1, 2, 3], "b": [4, 5, 6]})

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    df["a"][0] = 99                    # chained assignment
    print("after chained assignment:", df["a"].tolist(), "  <- unchanged!")
    print("warning raised:", [w.category.__name__ for w in caught])

In [ ]:
# The correct forms
df = pd.DataFrame({"a": [1, 2, 3], "b": [4, 5, 6]})
df.loc[0, "a"] = 99                      # single call, explicit
df.loc[df["b"] > 5, "a"] = 0             # conditional update - this is UPDATE ... WHERE
print(df)

In [ ]:
# A filtered frame is now always independent - no more guessing
base = pd.DataFrame({"a": [1, 2, 3], "b": [4, 5, 6]})
sub = base[base["a"] > 1]
sub.loc[:, "b"] = 0
print("sub:\n", sub)
print("base is untouched:\n", base)

## 5. Gotchas that produce wrong answers

These are the ones that do not raise an error.

### 5.1 `and` / `or` instead of `&` / `|`
Raises `ValueError` - annoying but safe. The dangerous version is forgetting parentheses: `df[df.a > 1 & df.b < 2]` parses as `df.a > (1 & df.b) < 2`.

### 5.2 Index alignment when assigning a Series

In [ ]:
df = pd.DataFrame({"x": [10, 20, 30]}, index=[0, 1, 2])
piece = pd.Series([1, 2, 3], index=[5, 6, 7])     # completely different labels

df["from_series"] = piece                          # aligned by LABEL -> all NaN
df["from_array"]  = piece.to_numpy()               # positional -> what you probably meant
print(df)
print()
print("Fix when you want positional: use .to_numpy(), or reset both indexes first.")

### 5.3 Comparing floats, and NaN != NaN

In [ ]:
a = pd.Series([0.1 + 0.2, 1.0])
b = pd.Series([0.3, 1.0])
print("a == b            :", (a == b).tolist())
print("np.isclose        :", np.isclose(a, b).tolist())
print("Series.equals     :", a.equals(b))
print()
n = pd.Series([1.0, np.nan])
print("n                 :", n.tolist())
print("n == n            :", (n == n).tolist(), " <- the NaN row is False: NaN never equals itself")
print("n.isna()          :", n.isna().tolist(), " <- this is the IS NULL test")

### 5.4 Duplicate column names

In [ ]:
dup = pd.DataFrame([[1, 2, 3]], columns=["a", "b", "a"])
print("df['a'] returns a DataFrame, not a Series:")
print(type(dup["a"]).__name__)
print()
print("check for this early:", dup.columns.duplicated().any())

### 5.5 A merge that multiplies rows

In [ ]:
left  = pd.DataFrame({"k": [1, 2], "v": ["a", "b"]})
right = pd.DataFrame({"k": [1, 1, 1, 2], "w": [10, 11, 12, 20]})
print("left rows:", len(left), " after merge:", len(left.merge(right, on="k")))
print("Always assert what you expect:")
try:
    left.merge(right, on="k", validate="1:1")
except Exception as e:
    print(" ", type(e).__name__, "-", str(e)[:60])

### 5.6 `inplace=True` is not an optimisation

It does not save memory in modern Pandas, it breaks method chaining, and several `inplace` paths are being removed. Prefer `df = df.something(...)`.

### 5.7 `groupby` drops NaN keys by default

In [ ]:
g = pd.DataFrame({"k": ["a", None, "a", "b"], "v": [1, 2, 3, 4]})
print("Pandas default: rows with a NaN key are DROPPED (SQL GROUP BY keeps NULL as its own group):")
print(g.groupby("k")["v"].sum())
print()
print("dropna=False keeps them - usually what you want when auditing data:")
print(g.groupby("k", dropna=False)["v"].sum())

### 5.8 Integer columns turn into floats

Covered in notebook 04: a single missing value promotes `int64` to `float64`. If downstream code writes those values back to a database as integers, you can get `1.0` where you expected `1`. Use the nullable `Int64` dtype when a column must stay integral.

### 5.9 `sort_values` is not stable by default

The default algorithm is quicksort. For reproducible ties, pass `kind="mergesort"` (stable) or add a tiebreaker column.

## 6. A practical optimisation checklist

Work down this list in order; stop when it is fast enough.

1. **Read less.** `usecols`, `nrows`, a Parquet file instead of CSV, push filters into the SQL query.
2. **Set correct dtypes at read time.** `dtype=`, `parse_dates=`, `category` for repeated text.
3. **Delete every Python loop.** Vectorized expressions, `np.select`, `np.where`, `Series.map`.
4. **Use `transform` instead of `apply`** for per-group calculations.
5. **Filter before joining and before aggregating,** not after.
6. **Pass `validate=` on merges** so you find row explosions immediately.
7. **Avoid `object` dtype.** Check with `df.dtypes`; `object` in a numeric column means something went wrong upstream.
8. **Profile before guessing.** `%timeit` in Jupyter, or the `timeit` helper at the top of this notebook.
9. **Only then** consider a different tool.

## 7. When to stop using Pandas

Pandas is single-threaded and holds everything in RAM, with a rule of thumb of about 5-10x the file size in memory. Rough guidance:

| Data size | Use |
|---|---|
| < 1 GB | Pandas |
| 1-50 GB on one machine | **Polars** (multi-threaded, lazy, Arrow-based) or **DuckDB** (run SQL directly over Parquet/CSV files) |
| bigger, or already in a warehouse | push the work into the database; only pull the aggregate into Pandas |
| genuinely distributed | **Spark** (PySpark), or **Dask** if you want to keep the Pandas API |

For a Java/Spring engineer the practical pattern is usually: aggregate in PostgreSQL, pull a small result into Pandas for analysis and charts. Do not stream a million rows into Python to sum them - that is the same mistake as pulling a table into an application to run a `COUNT`.

## 8. Exercises

**E1.** Rewrite this with no loop and time both:
```python
res = []
for _, r in orders.iterrows():
    res.append(r["amount"] * 0.82 if r["channel"] == "web" else r["amount"])
```

**E2.** Shrink `orders` to under half its current memory without losing precision on `amount`.

**E3.** Fix this so it actually updates the DataFrame: `orders[orders["amount"] > 5000]["status"] = "REVIEW"`

**E4.** Read only `order_id` and `amount` from `orders.csv` in chunks of 50 rows and find the maximum amount, without ever holding the whole file.

In [ ]:
# Solutions
print("E1:")
timeit(lambda: [r.amount * 0.82 if r.channel == "web" else r.amount
                for r in orders.itertuples(index=False)], 3, "  itertuples")
timeit(lambda: np.where(orders["channel"] == "web", orders["amount"] * 0.82, orders["amount"]),
       5, "  np.where")
print()

o2 = orders.copy()
for c in ["order_id", "customer_id", "discount_pct"]:
    o2[c] = pd.to_numeric(o2[c], downcast="integer")
for c in ["status", "channel"]:
    o2[c] = o2[c].astype("category")
print("E2:", f"{orders.memory_usage(deep=True).sum():,}", "->", f"{o2.memory_usage(deep=True).sum():,}",
      "bytes; amount dtype still", o2["amount"].dtype)
print()

o3 = orders.copy()
o3.loc[o3["amount"] > 5000, "status"] = "REVIEW"
print("E3: rows flagged:", (o3["status"] == "REVIEW").sum())
print()

running_max = -np.inf
for chunk in pd.read_csv(DATA / "orders.csv", usecols=["order_id", "amount"], chunksize=50):
    running_max = max(running_max, chunk["amount"].max())
print("E4: max amount =", running_max)

## Recap

- The ladder, slowest to fastest: `iterrows` -> `apply(axis=1)` -> `itertuples` -> `map` -> vectorized -> NumPy.
- Memory: read fewer columns, downcast integers, `category` for repeated text - but keep money in `float64` or integer minor units.
- Pandas 3.x: Copy-on-Write is always on. Chained assignment silently does nothing. Use `df.loc[mask, col] = value`.
- The silent-wrong-answer traps: index alignment on assignment, float equality, duplicate column names, many-to-many merges, `groupby` dropping NaN keys.
- Above a few GB, reach for Polars or DuckDB rather than fighting Pandas.

**Next:** `08_sql_java_cheatsheet.ipynb` - the reference tables and a set of practice problems.